# Data Analysis with Python — Class 07
## Missing Data Handling & Statistical Analysis — the two realities of real data

**Keynote Speaker:** Md Wabidur Rahman, Researcher, Research and Integrated Thought (RIT)
**Date:** 05 October 2026, Monday, 08:00 PM (Bangladesh Time)

---

### Today's roadmap

| Part | Topic | Keyword | In one line |
|------|------|----------|----------|
| 1 | **Spotting missing values** | `isnull`, `notnull` | finding where the blanks are |
| 2 | **Handling missing values** | `dropna`, `fillna`, `interpolate` | drop them or fill them |
| 3 | **Statistical analysis** | `describe`, `mean`, `corr` | from numbers to conclusions |

> **Link to the previous class:**
> In Class 06 we read `Shundorbon.xlsx` with Pandas and saw in `info()` that some cells were
> **blank (NaN)**. We said then that "missing values are very common in real data". Today that is the
> whole subject: **how to find and handle** missing values, and how to draw **statistics** from clean data.
> This is the largest part of a data analyst's daily work.

**How to run:** **Shift + Enter** in each cell. If any value looks strange, use **Kernel → Restart & Run All**.

---
# Part 1 — Reading the data and spotting the missing values

First job — read the file, attach the dates, and mark the blanks in a way Python understands.

In [ ]:
# importing the two libraries

import pandas as pd
import numpy as np

### Building the dates — `date_range()`

As in Class 06, `date_range()` builds a date for every day of 2020.
`freq='D'` means daily. 2020 was a leap year, so there are **366** dates in all.

In [ ]:
# a date for every day of 2020

date = pd.date_range('2020-01-01', '2020-12-31', freq='D')
print('total dates :', len(date))
date

### Reading the file and attaching the dates

`Khulna.csv` has a **14-line header** on top (NASA metadata), then the actual data.
- `skiprows=14` — drop the first fourteen lines
- `header=0` — take the next line as the column names
- `usecols=[3,4,5,6,7]` — keep only the five weather columns (dropping YEAR, MO, DY)
- `skipinitialspace=True` — strip the leading spaces from the column names

In [ ]:
# reading the file and making the date the index

data = pd.read_csv('Khulna.csv', skiprows=14, header=0,
                   usecols=[3, 4, 5, 6, 7], skipinitialspace=True)
data.index = date                 # one date per row

print('columns :', list(data.columns))
data.head(30)

### ⚠ Important — turning sentinel values into NaN

In a NASA file, a missing value is often marked with a **special number** (such as `-99999`).
Leave it in as a number and every mean will be wrong. So we convert it to Python's real
"missing" marker, **`NaN`** (Not a Number).

In [ ]:
#data.mean()

In [ ]:
# turning the sentinel -99999 into NaN

data = data.replace(-99999, np.nan)
data.head(30)

In [ ]:
# how many rows in all?

len(data)

# Identifying missing values — `isnull()` and `notnull()`

- **`isnull()`** — `True` for each cell that is blank, `False` where there is a value
- **`notnull()`** — exactly the opposite (`True` where there is a value)

These build a huge table of True/False on their own, so counting them with `.sum()` is far more
useful than looking at them directly.

In [ ]:
# where are the blanks? — the first ten rows

data.isnull().head(10)

In [ ]:
# how many blank cells in each column?
# (it adds them up, counting True as 1)

data.isnull().sum()

In [ ]:
# the other way round — how many values does each column have?

data.notnull().sum()

### At a glance — the total blanks and the percentages

How many cells are blank in the whole table, and what percentage in each column — this tells you
how big the problem is. It is what decides whether we **drop** rows or **fill** them.

In [ ]:
# the total blanks and the percentages

print('total blank cells in the table :', int(data.isnull().sum().sum()))
print()
print('percentage blank in each column:')
print((data.isnull().mean() * 100).round(2))

### Selecting only the non-blank values (in one column)

To take only the cells that hold a value from one column, we use `notnull()` as a filter —
just like the boolean indexing of Class 06.

In [ ]:
# only the filled cells of the Rainfall column

rain = data.Rainfall
d_rain = rain[rain.notnull()]       # only the non-blank values
print('rows in the original Rainfall :', len(rain))
print('rows with the blanks dropped  :', len(d_rain))
d_rain.head(10)

---
# Part 2 — Handling the missing values

Once you find missing values there are two paths —
1. **Drop them** (`dropna`) — throw away the rows or columns where they occur
2. **Fill them** (`fillna`, `interpolate`) — put a sensible value in their place

Which you choose depends on how much data is missing, and on whether filling it is defensible.

## Path 1 — dropping: `dropna()`

`dropna()` removes rows (or columns) that contain a blank. Its important options —
- **default** (`how='any'`) — drop the row if **even one** cell is blank
- **`how='all'`** — drop only if **every** cell in the row is blank
- **`thresh=n`** — keep the row if it has at least **n** values
- **`axis='columns'`** — drop blank-bearing columns instead of rows

In [ ]:
# always work on a copy so that the original data stays intact
# (without .copy() you may get a SettingWithCopyWarning)

data1 = data.copy()

# default: drop every row that has even one blank
print('rows before :', len(data1))
print('rows after dropna() :', len(data1.dropna(how='any')))
data1.dropna().head(30)

In [ ]:
# how='all' — drop only the completely blank rows

data2 = data.copy()
print('rows after dropna(how="all") :', len(data2.dropna(how='all')))
data2.dropna(how='all').head(10)

In [ ]:
# thresh=4 — keep the row if it has at least four values

data3 = data.copy()
kept = data3.dropna(thresh=4)
print('rows with at least four values :', len(kept))
kept.head(10)

## Path 2 — filling: `fillna()`

Dropping loses a lot of data, so filling is often better. The simplest form is to fill with a
single fixed value.

In [ ]:
# putting 0 in every blank cell

data4 = data.copy()
data4.fillna(0).head(30)

### ffill and bfill — pulling the value from before or after

For data like weather, putting 0 makes no sense (a temperature is hardly going to be 0 °C!). It is
better to take the value from a neighbouring day —
- **`ffill()`** (forward fill) — carries the **previous** day's value forward
- **`bfill()`** (backward fill) — carries the **next** day's value backward

In [ ]:
# forward fill — carrying the previous day's value forward

data5 = data.copy()
data5.ffill().head(30)

In [ ]:
# backward fill — carrying the next day's value backward

data6 = data.copy()
data6.bfill().head(30)

### Filling with the mean or the median — the most used in practice

In real analysis the blanks are often filled with the column's **mean** or **median**.
This preserves the overall character of the column.
- **mean** — ordinary, for well-behaved data
- **median** — safer when there are a few unusually large or small values (outliers)

In [ ]:
# filling each column's blanks with that column's mean

data7 = data.copy()
mean_filled = data7.fillna(data7.mean())

print('total blanks before filling :', int(data7.isnull().sum().sum()))
print('total blanks after filling  :', int(mean_filled.isnull().sum().sum()))
mean_filled.head(10)

In [ ]:
# an example of filling just one column with the median

print('mean of T_max   :', round(data.T_max.mean(), 2))
print('median of T_max :', round(data.T_max.median(), 2))

data8 = data.copy()
data8['T_max'] = data8['T_max'].fillna(data8['T_max'].median())
print()
print('blanks in T_max after filling :', int(data8['T_max'].isnull().sum()))

### `interpolate()` — a straight-line estimate between two values

The most intelligent option for time-based data. When there is a gap between two known values,
`interpolate()` estimates the middle **along a straight line**. If 5 January was 20° and
7 January was 22°, the estimate for 6 January is 21°.

In [ ]:
# linear interpolation

data9 = data.copy()
interp = data9.interpolate(method='linear')

print('total blanks after interpolate :', int(interp.isnull().sum().sum()))
interp.head(10)

### The three methods compared — on the same blank cell

Let us put the T_max value for one blank day (8 January, where every value was missing) side by side
under the three methods — that makes the difference clear.

In [ ]:
# T_max on 8 January — what value did each method put there?

compare = pd.DataFrame({
    'ffill (previous value)' : data.ffill()['T_max'],
    'bfill (next value)'     : data.bfill()['T_max'],
    'mean'                   : data.fillna(data.mean())['T_max'],
    'interpolate (linear)'   : data.interpolate()['T_max'],
})
compare.loc['2020-01-07':'2020-01-11'].round(2)

---
# Part 3 — Statistical analysis

Once the data is clean the real question is — **what are these numbers telling us?** Almost every
statistical function in Pandas computes **with the NaN excluded**, so they can be used with confidence.

In [ ]:
# describe() — the whole summary at a glance
# count, mean, std, min, 25%, 50%(median), 75%, max

data.describe().round(2)

> **How to read it:** `count` = how many values (blanks excluded), `mean` = the mean, `std` = the
> standard deviation (how spread out the values are), `min`/`max` = the lowest and highest,
> `50%` = the median.

In [ ]:
# the mean of each column

data.mean()

In [ ]:
# specific statistics for a single column

print('maximum rainfall :', data.Rainfall.max(), 'mm')
print('lowest maximum temperature (T_max min) :', data.T_max.min(), 'C')
print('median of T_min  :', data.T_min.median(), 'C')

### With NumPy too — `np.nanmean`

NumPy's ordinary `np.mean` returns `NaN` if there is a blank. So on data with blanks you need
**`np.nanmean`** — it excludes the NaN and takes the mean, exactly like Pandas' `.mean()`.

In [ ]:
# np.nanmean — checking it against Pandas' mean()

print('np.nanmean(T_max) :', np.nanmean(data.T_max))
print('data.T_max.mean() :', data.T_max.mean())

### A few more useful statistics

Beyond `describe()` there are several important measures —
- **`var()`** — the variance (the square of std)
- **`skew()`** — how far the distribution leans to one side (0 = symmetric, positive = a tail to the right)
- **`quantile(q)`** — the value below which a fraction q of the data falls (0.9 = the 90th percentile)

In [ ]:
# var, skew, quantile

print('variance:')
print(data.var().round(2))
print()
print('skew of the T_max distribution :', round(data.T_max.skew(), 3))
print('skew of the Rainfall distribution :', round(data.Rainfall.skew(), 3),
      '  <- strongly positive: heavy rain on a few days')
print()
print('90th percentile of T_max :', round(data.T_max.quantile(0.9), 2), 'C')

### Correlation — which column moves with which?

`corr()` shows how strongly two columns **rise and fall together**. The value runs from −1 to +1 —
- **close to +1** — when one rises the other rises too
- **close to 0** — no relationship
- **close to −1** — when one rises the other falls

In [ ]:
# the correlations among all the columns

data.corr().round(2)

> **An example of how to read it:** the relationship between T_min and Skin_temp is about **+0.94** —
> very strong, meaning that when the night temperature rises so does the surface temperature. And the
> relationship between T_max and Humidity is slightly **negative** — humidity is a little lower on hotter days.

### Summarising over time — `resample()`

This is where making the date the index pays off. `resample('M')` groups daily data **by month end**
(M = Month). Then `.mean()` gives the mean of each month — 366 rows collapse to just **12**.

In [ ]:
# the monthly means

monthly = data.resample('M').mean()
monthly.round(2)

In [ ]:
# from the monthly means — the hottest and wettest month of the year

print('hottest month (highest mean T_max) :',
      monthly.T_max.idxmax().strftime('%B'),
      '→', round(monthly.T_max.max(), 2), '°C')
print('wettest month (highest mean Rainfall) :',
      monthly.Rainfall.idxmax().strftime('%B'),
      '→', round(monthly.Rainfall.max(), 2), 'mm')

### Seeing it as a picture — temperature across the year

A picture explains faster than numbers. We draw the minimum and maximum temperature for the whole year
with `matplotlib`. The line breaks a little on the blank days — that is the natural signature of missing data.

In [ ]:
# T_min and T_max across the whole year — a line chart

import matplotlib.pyplot as plt

plt.figure(figsize=(12, 5))
plt.plot(data.index, data.T_min, label='Min temp', marker='o',
         markersize=3, color='#3D7EA6')
plt.plot(data.index, data.T_max, label='Max temp', marker='^',
         markersize=3, color='#B4562F')
plt.title('Khulna 2020 — Daily Min & Max Temperature')
plt.ylabel('Temperature (°C)')
plt.xlabel('Date')
plt.legend(loc='best')
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# the monthly means — a cleaner picture (less affected by the blanks)

plt.figure(figsize=(11, 5))
plt.plot(monthly.index.strftime('%b'), monthly.T_max, marker='^',
         label='Avg Max', color='#B4562F')
plt.plot(monthly.index.strftime('%b'), monthly.T_min, marker='o',
         label='Avg Min', color='#3D7EA6')
plt.title('Khulna 2020 — Monthly Average Temperature')
plt.ylabel('Temperature (°C)')
plt.legend(loc='best')
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

---
# Home Work

**1.** Read `Khulna.csv` (as we did today) and find how many blank cells each column has
(`isnull().sum()`), then print the **total** number of blank cells in the whole table.

**2.** Fill the blanks of the `Humidity` column once with the **mean** and once with the **median**.
Check in both cases that the number of blanks has fallen to 0.

**3.** Apply `interpolate()` to the whole dataset, then use `describe()` to see how much the
`count` and `mean` changed before and after the filling.

**4.** Use `resample('ME').mean()` to get the monthly means, and find in which month **Humidity was
highest**.

**5.** Draw a line chart (`matplotlib`) of the two columns `T_max` and `T_min`, with a title.

When the code runs correctly, take a **screenshot** and submit it.

---
# Today's summary

| Step | Job | Key functions |
|-----|-----|-----------|
| Spotting | finding and counting blanks | `isnull()` `notnull()` `.sum()` |
| Cleaning | sentinel -> NaN | `.replace(-99999, np.nan)` |
| Dropping | removing rows/columns | `dropna(how=, thresh=, axis=)` |
| Filling | putting values in | `fillna(0/mean/median)` `ffill()` `bfill()` `interpolate()` |
| Analysis | summary and relationships | `describe()` `mean` `var` `skew` `corr()` |
| Time | monthly means | `resample('ME').mean()` |

---

### The most common mistakes

1. **Forgetting the sentinel** — leave `-99999` unconverted and the mean, the max, everything is wrong. Use `.replace()` first
2. **Writing `fillna(method='ffill')`** — that is deprecated now; write `.ffill()` / `.bfill()` directly
3. **Assigning into a slice (`SettingWithCopyWarning`)** — take a `.copy()` at the start of the work
4. **Putting 0 in a blank temperature** — meaningless; use `mean`/`median`/`interpolate` instead
5. **`np.mean` versus `np.nanmean`** — on data with blanks `np.mean` returns only NaN; you need `np.nanmean`
---

**Md Wabidur Rahman** — Researcher, Research and Integrated Thought (RIT)
*Data Analysis with Python · Class 07 — Missing Data Handling & Statistical Analysis*